In [2]:
import torch
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("/sujin/PycharmProjects/Pretraining")

import numpy as np
import random
import json

from tqdm import tqdm
from Bio import SeqIO
from utils.fetch_uniprot import fetch_sequence
from utils.others import setup_seed
from utils.generate_lmdb import dump_lmdb, get_value
from dataset.ted.ted_dataset import TedDataset
from model.ted.ted_domain_model import TedDomainModel

In [4]:
path = "/sujin/Datasets/TED/ted_364m_domain_boundaries_consensus_level.tsv"
uniprot2domain = {}
with open(path, "r") as r:
    for line in tqdm(r):
        desc, span, _ = line.split("\t")
        domain_id = desc.split("_")[-1]
        uniprot_id = desc.split("-")[1]
        
        if uniprot_id not in uniprot2domain:
            uniprot2domain[uniprot_id] = []
        
        uniprot2domain[uniprot_id].append(span)

364806077it [14:23, 422618.05it/s]


In [ ]:
domain_cnt = {}
for spans in tqdm(uniprot2domain.values()):
    num_domain = len(spans)
    domain_cnt[num_domain] = domain_cnt.get(num_domain, 0) + 1

print(domain_cnt)

In [10]:
save_path = "/sujin/Datasets/TED/data_domain_ge2.tsv"
with open(save_path, "w") as w:
    for uniprot_id, spans in tqdm(uniprot2domain.items()):
        if len(spans) >= 2:
            w.write(f"{uniprot_id}\t{':'.join(spans)}\n")

100%|██████████| 203057497/203057497 [02:32<00:00, 1331968.96it/s]


In [4]:
domain_path = "/sujin/Datasets/TED/data_domain_ge2.tsv"
filter_uniprot2domain = {}
with open(domain_path, "r") as r:
    for line in tqdm(r):
        uniprot_id, spans = line.strip().split("\t")
        filter_uniprot2domain[uniprot_id] = {"domains": spans}

103868182it [02:04, 835878.42it/s]


# Load UniProt aa sequences

In [14]:
# We don't use SeqIO since it is slower
path = "/sujin/Datasets/ProTrek/TrEMBL/uniref100.fasta"
with open(path, "r") as r:
    uniprot_id = None
    for line in tqdm(r):
        content = line.strip()
        
        if content.startswith(">"):
            # Save previous seq
            if uniprot_id is not None and uniprot_id in filter_uniprot2domain:
                filter_uniprot2domain[uniprot_id]["aa_seq"] = seq

            uniprot_id = content.split(" ")[0].split("_")[-1]
            seq = ""

        else:
            seq += content
    
    # Add the last sequence
    if uniprot_id in filter_uniprot2domain:
        filter_uniprot2domain[uniprot_id]["aa_seq"] = seq

2501969694it [51:33, 808860.16it/s] 


# Load UniProt foldseek sequences

In [5]:
fasta_path = "/sujin/Datasets/ProTrek/foldseek_afdb/afdb_subset_ss.fasta"
with open(fasta_path, "r") as r:
    uniprot_id = None
    for line in tqdm(r):
        content = line.strip()
        
        if content.startswith(">"):
            # Save previous seq
            if uniprot_id is not None and uniprot_id in filter_uniprot2domain:
                filter_uniprot2domain[uniprot_id]["foldseek_seq"] =  seq

            uniprot_id = content.split("-")[1]
            seq = ""

        else:
            seq += content.lower()
    
    # Add the last sequence
    if uniprot_id in filter_uniprot2domain:
        filter_uniprot2domain[uniprot_id]["foldseek_seq"] = seq

429367658it [15:20, 466262.19it/s]


NameError: name 'uniprot2seq' is not defined

In [9]:
print(len(filter_uniprot2domain))

103868182


# Record protein information

In [25]:
domain_path = "/sujin/Datasets/TED/data_domain_ge2.tsv"
save_path = "/sujin/Datasets/TED/raw_seq_data_domain_ge2.tsv"
with open(save_path, "w") as w:
    for uniprot_id, data_dict in tqdm(filter_uniprot2domain.items()):
        domains = data_dict["domains"].strip()
        aa_seq = data_dict.get("aa_seq", "None")
        foldseek_seq = data_dict.get("foldseek_seq", "None")
        w.write(f"{uniprot_id}\t{domains}\t{aa_seq}\t{foldseek_seq}\n")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103868182/103868182 [05:23<00:00, 321081.64it/s]


# Statistics before filtering

In [29]:
no_seq_cnt = 0
only_aa_cnt = 0
only_foldseek_cnt = 0
aa_foldseek_mismatch_cnt = 0
normal_cnt = 0

for uniprot_id, data_dict in tqdm(filter_uniprot2domain.items()):
    domains = data_dict["domains"].strip()
    aa_seq = data_dict.get("aa_seq", None)
    foldseek_seq = data_dict.get("foldseek_seq", None)
    
    if aa_seq is None and foldseek_seq is None:
        no_seq_cnt += 1
    
    elif aa_seq is None:
        only_foldseek_cnt += 1
    
    elif foldseek_seq is None:
        only_aa_cnt += 1
    
    else:
        if len(aa_seq) != len(foldseek_seq):
            aa_foldseek_mismatch_cnt += 1
        
        else:
            normal_cnt += 1

print("no_seq_cnt", no_seq_cnt)
print("only_aa_cnt", only_aa_cnt)
print("only_foldseek_cnt", only_foldseek_cnt)
print("aa_foldseek_mismatch_cnt", aa_foldseek_mismatch_cnt)
print("normal_cnt", normal_cnt)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103868182/103868182 [03:36<00:00, 480848.00it/s]

no_seq_cnt 0
only_aa_cnt 0
only_foldseek_cnt 18091470
aa_foldseek_mismatch_cnt 21341
normal_cnt 85755371


In [6]:
seq_path = "/sujin/Datasets/TED/raw_seq_data_domain_ge2.tsv"
incomplete_ids = {}
with open(seq_path, "r") as r:
    for line in tqdm(r):
        uniprot_id, domains, aa_seq, foldseek_seq = line.strip().split("\t")
        
        if aa_seq == "None" and foldseek_seq == "None":
            incomplete_ids[uniprot_id] = "No having any sequence"
        
        elif aa_seq == "None":
            incomplete_ids[uniprot_id] = "Only having foldseek sequence"
        
        elif foldseek_seq == "None":
            incomplete_ids[uniprot_id] = "Only having aa sequence"
        
        else:
            if len(aa_seq) != len(foldseek_seq):
                incomplete_ids[uniprot_id] = "Mismatch between aa sequence and foldseek sequence"

103868182it [08:16, 209059.97it/s]


In [ ]:
save_path = "/sujin/Datasets/TED/incomplete_ids.tsv"
with open(save_path, "w") as w:
    for uniprot_id, reason in tqdm(incomplete_ids.items()):
        w.write(f"{uniprot_id}\n")

# Repair incomplete sequences

In [4]:
seq_path = "/sujin/Datasets/TED/raw_seq_data_domain_ge2.tsv"
uniprot2data = {}
with open(seq_path, "r") as r:
    for line in tqdm(r):
        uniprot_id, domains, aa_seq, foldseek_seq = line.strip().split("\t")
        uniprot2data[uniprot_id] = {
            "domains": domains,
            "aa_seq": aa_seq,
            "foldseek_seq": foldseek_seq
        }

103868182it [05:06, 339192.97it/s]


In [9]:
afdb_fasta_path = "/sujin/Datasets/FoldseekDB/afdb/afdb_aa.fasta"
with open(afdb_fasta_path, "r") as r:
    uniprot_id = None
    for line in tqdm(r):
        content = line.strip()
        
        if content.startswith(">"):
            # Save previous seq
            if uniprot_id is not None and uniprot_id in uniprot2data:
                uniprot2data[uniprot_id]["aa_seq"] = seq

            uniprot_id = content.split("-")[1]
            seq = ""

        else:
            seq += content
    
    # Add the last sequence
    if uniprot_id in uniprot2data:
        uniprot2data[uniprot_id]["aa_seq"] = seq

429367658it [13:28, 530995.97it/s]


# Statistics after filtering

In [10]:
no_seq_cnt = 0
only_aa_cnt = 0
only_foldseek_cnt = 0
aa_foldseek_mismatch_cnt = 0
normal_cnt = 0

for uniprot_id, data_dict in tqdm(uniprot2data.items()):
    domains = data_dict["domains"].strip()
    aa_seq = data_dict.get("aa_seq", None)
    foldseek_seq = data_dict.get("foldseek_seq", None)
    
    if aa_seq is None and foldseek_seq is None:
        no_seq_cnt += 1
    
    elif aa_seq is None:
        only_foldseek_cnt += 1
    
    elif foldseek_seq is None:
        only_aa_cnt += 1
    
    else:
        if len(aa_seq) != len(foldseek_seq):
            aa_foldseek_mismatch_cnt += 1
        
        else:
            normal_cnt += 1

print("no_seq_cnt", no_seq_cnt)
print("only_aa_cnt", only_aa_cnt)
print("only_foldseek_cnt", only_foldseek_cnt)
print("aa_foldseek_mismatch_cnt", aa_foldseek_mismatch_cnt)
print("normal_cnt", normal_cnt)

100%|██████████| 103868182/103868182 [04:09<00:00, 415579.43it/s]

no_seq_cnt 0
only_aa_cnt 0
only_foldseek_cnt 0
aa_foldseek_mismatch_cnt 0
normal_cnt 103868182


In [12]:
domain_path = "/sujin/Datasets/TED/data_domain_ge2.tsv"
save_path = "/sujin/Datasets/TED/complete_seq_data_domain_ge2.tsv"
with open(save_path, "w") as w:
    for uniprot_id, data_dict in tqdm(uniprot2data.items()):
        domains = data_dict["domains"].strip()
        aa_seq = data_dict["aa_seq"]
        foldseek_seq = data_dict["foldseek_seq"]
        w.write(f"{uniprot_id}\t{domains}\t{aa_seq}\t{foldseek_seq}\n")

100%|██████████| 103868182/103868182 [04:39<00:00, 371590.24it/s]


# Cluster proteins based on AFDB cluster

In [20]:
seq_path = "/sujin/Datasets/TED/raw_seq_data_domain_ge2.tsv"
uniprot_ids = set()
with open(seq_path, "r") as r:
    for line in tqdm(r):
        uniprot_id, domains, aa_seq, foldseek_seq = line.strip().split("\t")
        if aa_seq == "None" or foldseek_seq == "None" or len(aa_seq) != len(foldseek_seq):
            continue
        
        uniprot_ids.add(uniprot_id)

103868182it [09:19, 185637.10it/s]


In [14]:
uniprot_ids = set(uniprot2data.keys())

KeyboardInterrupt: 

In [2]:
# seq_path = "/sujin/Datasets/TED/raw_seq_data_domain_ge2.tsv"
# uniprot_ids = set()
# with open(seq_path, "r") as r:
#     for line in tqdm(r):
#         uniprot_id, domains, aa_seq, foldseek_seq = line.strip().split("\t")
#         if aa_seq == "None" or foldseek_seq == "None" or len(aa_seq) != len(foldseek_seq):
#             continue
#         
#         uniprot_ids.add(uniprot_id)

seq_path = "/sujin/Datasets/TED/complete_seq_data_domain_ge2.tsv"
plddt_lmdb = "/sujin/Datasets/LMDB/afdb_plddt"
uniprot2data = {}
with open(seq_path, "r") as r:
    for line in tqdm(r):
        try:
            uniprot_id, domains, aa_seq, foldseek_seq = line.strip().split("\t")
            plddt_dict = json.loads(get_value(plddt_lmdb, uniprot_id))
            plddt_list = json.loads(f"[{plddt_dict['plddt']}]")
            avg_plddt = np.mean(plddt_list)
            
            if avg_plddt >= 70:
                uniprot2data[uniprot_id] = {
                    "domains": domains,
                    "aa_seq": aa_seq,
                    "foldseek_seq": foldseek_seq
                }
        
        except Exception:
            print(uniprot_id)
            pass

7837699it [25:50, 5277.80it/s]

A0A0W8IES4


8264782it [27:14, 4867.32it/s]

A0A118JXS4


9911799it [32:42, 5315.52it/s]

...

94761375it [5:52:58, 2650.98it/s]

A0A1Y5EZP1


103764312it [7:15:29, 2851.21it/s]

J0L0Z0


103868182it [7:16:07, 3969.36it/s]


In [3]:
len(uniprot2data)

92646632

In [4]:
cluster_tsv = "/sujin/Datasets/TED/5-allmembers-repId-entryId-cluFlag-taxId.tsv"
cluster2member = {}
with open(cluster_tsv, "r") as r:
    for line in tqdm(r):
        rep_id, mem_id = line.strip().split("\t")[:2]
        if mem_id in uniprot2data:
            if rep_id not in cluster2member:
                cluster2member[rep_id] = []

            cluster2member[rep_id].append(mem_id)

214684311it [05:18, 674651.92it/s] 


In [5]:
# Check whether every protein has a cluster
cnt = 0
for cluster_id, members in tqdm(cluster2member.items()):
    cnt += len(members)

assert cnt == len(uniprot2data)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1957679/1957679 [00:00<00:00, 2350344.05it/s]


# Sample validation and test set

In [7]:
cluster_ids = list(cluster2member.keys())
print(len(cluster_ids))

1957679


In [8]:
setup_seed(20000812)

n_cluster = 2000
sampled_ids = random.sample(cluster_ids, n_cluster)

path = "/sujin/Datasets/TED/splits/test_20260215.tsv"
test_cnt = 0
with open(path, "w") as w:
    for sampled_id in tqdm(sampled_ids[:1000]):
        members = cluster2member[sampled_id]
        for member in members:
            test_cnt += 1
            w.write(f"{sampled_id}\t{member}\n")
print(test_cnt)

path = "/sujin/Datasets/TED/splits/valid_20260215.tsv"
valid_cnt = 0
with open(path, "w") as w:
    for sampled_id in tqdm(sampled_ids[1000:]):
        members = cluster2member[sampled_id]
        for member in members:
            valid_cnt += 1
            w.write(f"{sampled_id}\t{member}\n")
print(valid_cnt)            

train_cnt = 0
sampled_id_set = set(sampled_ids)
path = "/sujin/Datasets/TED/splits/train_20260215.tsv"
with open(path, "w") as w:
    for cluster_id, members in tqdm(cluster2member.items()):
        if cluster_id not in sampled_id_set:
            for member in members:
                train_cnt += 1
                w.write(f"{cluster_id}\t{member}\n")
print(train_cnt)

print(len(uniprot2data))
print(test_cnt+valid_cnt+train_cnt)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 65795.07it/s]


49679


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 91781.09it/s]


34296


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1957679/1957679 [00:25<00:00, 75333.38it/s]

92562657
92646632
92646632


# Construct LMDB dataset

In [3]:
seq_path = "/sujin/Datasets/TED/complete_seq_data_domain_ge2.tsv"
plddt_lmdb = "/sujin/Datasets/LMDB/afdb_plddt"
uniprot2data = {}
with open(seq_path, "r") as r:
    for line in tqdm(r):
        uniprot_id, domains, aa_seq, foldseek_seq = line.strip().split("\t")
        uniprot2data[uniprot_id] = {
            "domains": domains,
            "aa_seq": aa_seq,
            "foldseek_seq": foldseek_seq
        }

103868182it [05:28, 316584.54it/s]


In [11]:
plddt_lmdb = "/sujin/Datasets/LMDB/afdb_plddt"
for uniprot_id in tqdm(uniprot2data):
    get_value(plddt_lmdb, uniprot_id)

  0%|          | 102603/103868182 [00:12<3:28:21, 8300.42it/s]


KeyboardInterrupt: 

In [4]:
def domain2seq(domains: str, aa_seq: str, foldseek_seq: str) -> list:
    sa_segs = []
    for domain in domains.split(":"):
        # If a domain is discontinuous
        if "_" in domain:
            sa_part_list = []
            for part in domain.split("_"):
                st, ed = part.split("-")
                aa_seg_part = aa_seq[int(st)-1: int(ed)]
                foldseek_seg_part = foldseek_seq[int(st)-1: int(ed)]
                sa_seg_part = "".join(aa + _3di for aa, _3di in zip(aa_seg_part, foldseek_seg_part))
                
                sa_part_list.append(sa_seg_part)
                
            sa_seg = "<unk>".join(sa_part_list) 
        
        else:
            st, ed = domain.split("-")
            aa_seg_part = aa_seq[int(st)-1: int(ed)]
            foldseek_seg_part = foldseek_seq[int(st)-1: int(ed)]
            sa_seg = "".join(aa + _3di for aa, _3di in zip(aa_seg_part, foldseek_seg_part))
        
        sa_segs.append(sa_seg)
    
    return sa_segs


domains = "11-41_290-389:54-288"
aa_seq = "MDFFVRLARETGDRKREFLELGRKAGRFPAASTSNGEISIWCSNDYLGMGQHPDVLDAMKRSVDEYGGGSGGSRNTGGTNHFHVALEREPAEPHGKEDAVLFTSGYSANEGSLSVLAGAVDDCQVFSDSANHASIIDGLRHSGARKHVFRHKDGRHLEELLAAADRDKPKFIALESVHSMRGDIALLAEIAGLAKRYGAVTFLDEVHAVGMYGPGGAGIAARDGVHCEFTVVMGTLAKAFGMTGGYVAGPAVLMDAVRARARSFVFTTALPPAVAAGALAAVRHLRGSDEERRRPAENARLTHGLLRERDIPVLSDRSPIVPVLVGEDRMCKRMSALPLERHGAYVQAIDAPSVPAGEEILRIAPSAVHETEEIHRFVDALDGIWSELGAARRV"
foldseek_seq = "dvvvvvvvvvcvvvddddfdwdcdqvdppwidgpvgifgeffallqlsllrdpllvvllvvlcvvpnffqpfapvpsnddplqvllfpllqvllvfptkdwfqalllqllllllqpqqvfpqeeeeeeplddpsnvnspvnsvhhydyayhldlvsvlvvlvvddlphayeyedeqardaqrdghpvvsvlvscvvsvyayeyeqasqalqdaqlrsgdcnnvvnsvshakykyfcvgnvsagiimighypvsnvsscvgrcsrgvdrdggssrssssssssvvsnvdplsnvqllvqlvllvvlcvvllfdfsdsnhswtwgwqqadvllvqlqvqccvpvsyhwdwddppnddrrritgtgrrhssddpvnsvvssvsssvscvvsvtdghd"

sa_segs = domain2seq(domains, aa_seq, foldseek_seq)
sa_segs

['TcGvDvRvKdRdEdFdLfEdLwGdRcKdAqGvRdFpPpAwAiSdTgSpNvGgEiIfSgIeWf<unk>ElEsRnRvRqPlAlEvNqAlRvLlTlHvGvLlLcRvEvRlDlIfPdVfLsSdDsRnShPsIwVtPwVgLwVqGqEaDdRvMlClKvRqMlSqAvLqPcLcEvRpHvGsAyYhVwQdAwIdDdApPpSnVdPdArGrErEiItLgRtIgArPrShAsVsHdEdTpEvEnIsHvRvFsVsDvAsLsDsGvIsWcSvEvLsGv',
 'DlVlLvDvAlMlKvRvSlVcDvEvYpGnGfGfSqGpGfSaRpNvTpGsGnTdNdHpFlHqVvAlLlEfRpElPlAqEvPlHlGvKfEpDtAkVdLwFfTqSaGlYlSlAqNlElGlSlLlSlVqLpAqGqAvVfDpDqCeQeVeFeSeDeSpAlNdHdApSsInIvDnGsLpRvHnSsGvAhRhKyHdVyFaRyHhKlDdGlRvHsLvElEvLvLlAvAvAdDdRlDpKhPaKyFeIyAeLdEeSqVaHrSdMaRqGrDdIgAhLpLvAvEsIvAlGvLsAcKvRvYsGvAyVaTyFeLyDeEqVaHsAqVaGlMqYdGaPqGlGrAsGgIdAcAnRnDvGvVnHsCvEsFhTaVkVyMkGyTfLcAvKgAnFvGsMaTgGiGiYmViAgGhPyApVvLsMnDvAsVsRcAvRgArRcSsFrVgFvTdTrAdLgPgPsAsVrAsAsGsAsLsAsAsVsRvHvLsRnGvSd']

# Compute the number of duplicated segments

In [5]:
seq_path = "/sujin/Datasets/TED/complete_seq_data_domain_ge2.tsv"
save_path = "/sujin/Datasets/TED/sa_segments_ge2.tsv"
with open(save_path, "w") as w, open(seq_path, "r") as r:
    for line in tqdm(r):
        uniprot_id, domains, aa_seq, foldseek_seq = line.strip().split("\t")
        sa_segs = domain2seq(domains, aa_seq, foldseek_seq)
        
        for sa_seg in sa_segs:
            w.write(f"{uniprot_id}\t{sa_seg}\n")

103868182it [1:04:49, 26702.89it/s]


# Split segments into sequence part and 3Di part

In [4]:
seg_path = "/sujin/Datasets/TED/sa_segments_ge2.tsv"
fasta_path = "/sujin/Datasets/TED/aa_segments_ge2/aa_segments_ge2.fasta"
_3di_path = "/sujin/Datasets/TED/3di_segments_ge2/3di_segments_ge2.fasta"
with open(fasta_path, "w") as aa_w, open(_3di_path, "w") as _3di_w, open(seg_path, "r") as r:
    cnt = 0
    for i, line in enumerate(tqdm(r)):
        uniprot_id, sa_seg = line.strip().split("\t")
        sa_seg = sa_seg.replace("<unk>", "")
        aa_seq = sa_seg[::2]
        _3di_seq = sa_seg[1::2].upper()
        aa_w.write(f">{i}\n{aa_seq}\n")
        _3di_w.write(f">{i}\n{_3di_seq}\n")
        # cnt += 1
        # if cnt % 10000 == 0:
        #     break

265616762it [12:42, 348218.74it/s]


# Check the overlap among training, validation, and test sets

In [3]:
seg_path = "/sujin/Datasets/TED/sa_segments_ge2.tsv"
uniprot2idx = {}
idx2uniprot = {}
idx2seg = {}
seg2idx = {}
with open(seg_path, "r") as r:
    for line in tqdm(r):
        uniprot_id, sa_seg = line.strip().split("\t")
        idx = seg2idx.get(sa_seg, len(seg2idx))
        seg2idx[sa_seg] = idx
        idx2seg[idx] = sa_seg
        uniprot2idx[uniprot_id] = uniprot2idx.get(uniprot_id, []) + [idx]
        idx2uniprot[idx] = idx2uniprot.get(idx, []) + [uniprot_id]

265616762it [16:36, 266683.04it/s]


In [8]:
stage2inds = {
    "train": set(),
    "valid": set(),
    "test": set()
}
for stage in ["train", "valid", "test"]:
    tsv_path = f"/sujin/Datasets/TED/splits/{stage}_20251204.tsv"
    with open(tsv_path, "r") as r:
        for line in tqdm(r):
            rep_id, mem_id = line.strip().split("\t")
            inds = set(uniprot2idx[mem_id])
            for idx in inds:
                stage2inds[stage].add(idx)

92562657it [04:27, 346250.95it/s]
34296it [00:00, 286942.19it/s]
49679it [00:00, 340285.67it/s]


In [9]:
print("Train set size:", len(stage2inds["train"]))
print("Validation set size:", len(stage2inds["valid"]))
print("Test set size:", len(stage2inds["test"]))

# Remove UniProt proteins that have duplicated domains in the training set
for stage in ["test", "valid"]:
    dup_uniprot_ids = set()
    for idx in (stage2inds["train"] & stage2inds[stage]):
        dup_uniprot_ids.update(idx2uniprot[idx])
    
    ori_tsv_path = f"/sujin/Datasets/TED/splits/{stage}_20251204.tsv"
    new_tsv_path = f"/sujin/Datasets/TED/splits/{stage}_20251204_no_dup.tsv"
    with open(ori_tsv_path, "r") as r, open(new_tsv_path, "w") as w:
        for line in tqdm(r):
            rep_id, mem_id = line.strip().split("\t")
            if mem_id not in dup_uniprot_ids:
                w.write(line)

Train set size: 232155978
Validation set size: 114530
Test set size: 110885


49679it [00:00, 822886.41it/s]
34296it [00:00, 808129.45it/s]


In [21]:
for stage in ["valid_20251204_no_dup", "test_20251204_no_dup"]:
    tsv_path = f"/sujin/Datasets/TED/splits/{stage}.tsv"
    lmdb_dir = f"/sujin/Datasets/LMDB/TED/only_2domain_plddt70/{stage.split('_')[0]}"

    with open(tsv_path, "r") as r:
        # LMDB data
        data_dict = {}
        
        # Used to record unique domains in the dataset
        domain2idx = {}
        
        cnt = 0
        for line in tqdm(r):
            rep_id, mem_id = line.strip().split("\t")
            domain_inds = uniprot2idx[mem_id]
            
            # Filter out proteins with more than 2 domains
            if len(domain_inds) != 2:
                continue
            
            sa_segs = [idx2seg[domain_idx] for domain_idx in domain_inds]
            for sa_seg in sa_segs:
                if sa_seg not in domain2idx:
                    idx = len(domain2idx)
                    domain2idx[sa_seg] = f"domain_{idx}"
                    data_dict[f"domain_{idx}"] = sa_seg
            
            # Construct pairwise data
            for i in range(len(sa_segs)):
                for j in range(i+1, len(sa_segs)):
                    sa_seg_1 = sa_segs[i]
                    sa_seg_2 = sa_segs[j]
                    
                    key_1 = domain2idx[sa_seg_1]
                    key_2 = domain2idx[sa_seg_2]
                    data_dict[cnt] = f"{key_1}\t{key_2}"
                    # print(f"{key_1}\t{key_2}")
                    cnt += 1
                    if cnt % 100000 == 0:
                        dump_lmdb(data_dict, lmdb_dir, verbose=False)
                        data_dict = {}
        
        data_dict["length"] = str(cnt)
        data_dict["num_domains"] = str(len(domain2idx))
        dump_lmdb(data_dict, lmdb_dir)

33734it [00:00, 181567.50it/s]
Dumping data...: 100%|██████████| 45829/45829 [00:00<00:00, 595057.28it/s]
49407it [00:00, 140562.35it/s]
Dumping data...: 100%|██████████| 122383/122383 [00:00<00:00, 561802.35it/s]


60090343
domain_117592650	domain_117592651
ClHvSlLlNqLvTlLlCvDlMlAcQvSlCdTpKcAsIvScFlFvGvVlIlQqRlIlYvVcLlFqSvSvSdTpKvRlWvKvIlLlLcDvNlVqPvKpLaTdVaKdAhLdSdTpTrRdWlEvSnRvIlKsSnVlQvAsVcRlFvQrTvPvQsIsRlDvAsLlMvEvVqGlRvCvSpTpNpDpPpKpGsLnSvDsAsQvGvLsVnTvAlLsElNaFlElFsLqClGvMsVvIlWvHnDvIsLsFvSlIsNvMvVlSsKvKvLsQpSdKpTpVdCdMlDvVnAnLvEvQsIlKvGvVvIlLvYvFlKvKvYcRlAvEpGvFlSvRvIsIlDvIsAsEvEvMsDpVhEdPnVfFhPd<unk>ArLvSvArIsEvScFsRsVvNrYrFvLnVsMsIsDvMsAsInAvSsLsNcSvRsFcEvQvMsEvIvFcEcNqIq


In [22]:
# lmdb_dir = "/sujin/Datasets/LMDB/TED/preliminary/test"
# print(get_value(lmdb_dir, "length"))
# print(get_value(lmdb_dir, "num_domains"))
# print(get_value(lmdb_dir, "0"))
# print(get_value(lmdb_dir, "1"))
# print(get_value(lmdb_dir, "2"))
# print(get_value(lmdb_dir, "3"))
# # print(get_value(lmdb_dir, "domain_0"))
# # print(get_value(lmdb_dir, "domain_1"))
#
# print(get_value(lmdb_dir, "domain_5351").replace("<unk>", "")[::2])
# print(get_value(lmdb_dir, "domain_5353").replace("<unk>", "")[::2])
# print(get_value(lmdb_dir, "domain_5149").replace("<unk>", "")[::2])
# print(get_value(lmdb_dir, "domain_5157").replace("<unk>", "")[::2])
# print(get_value(lmdb_dir, "domain_5151").replace("<unk>", ""))

lmdb_dir = "/sujin/Datasets/LMDB/TED/only_2domain_plddt70/test"
print(get_value(lmdb_dir, "length"))
print(get_value(lmdb_dir, "num_domains"))
print(get_value(lmdb_dir, "2"))
print(get_value(lmdb_dir, "domain_2"))

40898
81483
domain_4	domain_5
QvHsFcScSvVqYaSvKpPlIlLvAlSlRvGdApKpChTdPpAvQsAvEvAsGnAvLvAsLvEvAvLvHcRvQsVcLvPvFvMd


In [21]:
from Bio.Align import PairwiseAligner

def calc_seq_identity(seq1: str, seq2: str) -> float:
    try:
        aligner = PairwiseAligner()
        aligner.mode = "local"
    
        alignment = next(aligner.align(seq1, seq2))
        a1, a2 = alignment
        identity = sum(1 for a, b in zip(a1, a2) if a == b) / len(a1)
        return identity
    
    except Exception as e:
        print(e)
        return 0.0

seq_1 = get_value(lmdb_dir, "domain_1").replace("<unk>", "")[::2]
seq_2 = get_value(lmdb_dir, "domain_5").replace("<unk>", "")[::2]
print(calc_seq_identity(seq_1, seq_2))

0.21991701244813278


# Ted Dataset 

In [48]:
config = {
    "tokenizer": "/sujin/Models/SaProt/SaProt_650M_AF2",
    "train_lmdb": "/sujin/Datasets/LMDB/TED/preliminary/test",
    "max_length": 1024
}

ted_dataset = TedDataset(**config)
ted_dataset._init_lmdb(ted_dataset.train_lmdb)
ted_dataset.__getitem__(2)

domain_4 domain_5


('LvSvIlVlHvSlLlMqCvHqRfQpGdGlEdSdEsSvFqAsKsRqAlIsEsSvLlVnKvKqLcKsEvKvRrDqElLsDrSlLlIsTcAcIqTvTvNvGnSpHdHdSgKaCfVnTkIdQfRaTdLpDvGcRwLdQdVtApGnRdKtGdFhPlHlVqIvYlAlRcIgWrRpWnPsDpLdHdKpNvEqLkKdHfVdKpYpChQpFaAdFsDvLvKvCdDrSmVhCgVnNrPnYsHrYiEdRgVh',
 'WqCkSwIkAwYkFdEfLqDqTdQtVwGdEdTiFqKtVaSgSpSsCaPqNkVaTwVeDaGlYdVqDdPdSaGdAnNrRyFhChLpGvAvLiSdNdVpHpRaTdEpQqSlDvRvAlRsLvHqIcGhKsGpVkQmLwDgVaRdGdEnGgDwViWkLiRfCaLqStDqHaSkVkFkVkQqSaYqYvLvDcRvEvAvGvRhSdPrGrDrAhViHyKiIyYhPhSgAyYmIdKtVrFaDdLlRvQvClHlGvQvMlQl<unk>DnLnRqRvLnCqIkLmRkLmSaFgVrKdGdWaGpPpDvYdPpRhQpSdIpKsErTgPrCiWiImEiVmHgLrHvRvAsLvQvLsLnDvEvVsLv')

# Ted Model

In [51]:
config = {
    "config_path": "/sujin/Models/SaProt/SaProt_650M_AF2",
}

ted_model = TedDomainModel(**config)
device = "cuda"
ted_model.to(device)

TedDomainModel(
  (model): EsmForMaskedLM(
    (esm): EsmModel(
      (embeddings): EsmEmbeddings(
        (word_embeddings): Embedding(446, 1280, padding_idx=1)
        (dropout): Dropout(p=0.1, inplace=False)
        (position_embeddings): None
      )
      (encoder): EsmEncoder(
        (layer): ModuleList(
          (0-32): 33 x EsmLayer(
            (attention): EsmAttention(
              (self): EsmSelfAttention(
                (query): Linear(in_features=1280, out_features=1280, bias=True)
                (key): Linear(in_features=1280, out_features=1280, bias=True)
                (value): Linear(in_features=1280, out_features=1280, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
                (rotary_embeddings): RotaryEmbedding()
              )
              (output): EsmSelfOutput(
                (dense): Linear(in_features=1280, out_features=1280, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (La

In [69]:
setup_seed(20000812)

with torch.no_grad():
    for dataloader in ted_dataset.train_dataloader():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            output = ted_model(**batch)
            loss = ted_model.loss_func("train", output, labels={})
            break
        break

RuntimeError: TedDomainModel is not attached to a `Trainer`.